In [1]:
# ========================================================================================================
# THEME 3 — CELLULE "STORY LOADER" : Recharge modèles + preuve d'itérations + rapport d'étonnement (auto)
# ========================================================================================================
#
# BUT :
# - Recharger et inventorier tous les modèles/checkpoints/ONNX produits au fil des thèmes
# - Fournir une preuve "exécutable" de l'évolution itérative (baseline -> TL best -> champion -> retrained)
# - Générer une base "lessons learned / rapport d'étonnement" à partir des artefacts observés
#
# Dossiers attendus :
# - WORK_DIR/checkpoints
# - WORK_DIR/artifacts (dont *.onnx + champion_config.json)
# ========================================================================================================

import json
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from collections import OrderedDict
from datetime import datetime

# --------------------------------------------
# 0) CONFIG
# --------------------------------------------
WORK_DIR = Path(r"C:\Users\Yanis\IA_GEN_PROJET")
CKPT_DIR = WORK_DIR / "checkpoints"
ART_DIR  = WORK_DIR / "artifacts"

print(f"📁 WORK_DIR : {WORK_DIR}")
print(f"📦 CKPT_DIR : {CKPT_DIR} (exists={CKPT_DIR.exists()})")
print(f"🧪 ART_DIR  : {ART_DIR} (exists={ART_DIR.exists()})")

# --------------------------------------------
# 1) Helpers
# --------------------------------------------
def build_densenet121_1ch_2cls():
    m = models.densenet121(weights=None)
    m.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.classifier = nn.Linear(m.classifier.in_features, 2)
    return m

def describe_checkpoint(obj):
    info = {"type": str(type(obj))}
    if isinstance(obj, OrderedDict):
        keys = list(obj.keys())
        info["format"] = "OrderedDict(state_dict)"
        info["n_keys"] = len(keys)
        info["top_keys"] = keys[:10]
    elif isinstance(obj, dict):
        keys = list(obj.keys())
        info["format"] = "dict(checkpoint)"
        info["n_keys"] = len(keys)
        info["top_keys"] = keys[:12]
        # best-effort detect state_dict field
        for k in ["model_state_dict", "state_dict", "model"]:
            if k in obj and isinstance(obj[k], (dict, OrderedDict)):
                info["state_dict_field"] = k
                info["state_dict_keys"] = len(obj[k].keys())
                break
    else:
        info["format"] = "unknown"
    return info

def safe_load_state_dict(model, state_dict):
    # Strict first, fallback strict=False to inspect mismatches
    try:
        model.load_state_dict(state_dict, strict=True)
        return {"ok": True, "missing": 0, "unexpected": 0}
    except Exception as e:
        res = model.load_state_dict(state_dict, strict=False)
        return {"ok": False, "error": repr(e),
                "missing": len(res.missing_keys),
                "unexpected": len(res.unexpected_keys),
                "missing_preview": res.missing_keys[:5],
                "unexpected_preview": res.unexpected_keys[:5]}

def load_model_from_path(path: Path):
    obj = torch.load(path, map_location="cpu")
    meta = describe_checkpoint(obj)

    model = build_densenet121_1ch_2cls()

    # extract state_dict
    if isinstance(obj, OrderedDict):
        state_dict = obj
        load_res = safe_load_state_dict(model, state_dict)
    elif isinstance(obj, dict):
        if "model_state_dict" in obj and isinstance(obj["model_state_dict"], (dict, OrderedDict)):
            state_dict = obj["model_state_dict"]
        elif "state_dict" in obj and isinstance(obj["state_dict"], (dict, OrderedDict)):
            state_dict = obj["state_dict"]
        else:
            # best-effort: consider dict itself is a state_dict, but filter out known training keys
            blacklist = {"epoch","phase","train_loss","train_acc","val_loss","val_acc","optimizer_state_dict","metrics"}
            state_dict = {k:v for k,v in obj.items() if k not in blacklist and isinstance(v, torch.Tensor)}
        load_res = safe_load_state_dict(model, state_dict)
    else:
        state_dict = None
        load_res = {"ok": False, "error": "Unsupported checkpoint object"}

    return model, meta, load_res

def file_exists(p): return "✅" if p.exists() else "❌"

# --------------------------------------------
# 2) Candidate artifacts discovery
# --------------------------------------------
# Tu peux adapter cette liste si tes noms changent
candidates = [
    ("T3_history_TL_pt", CKPT_DIR / "history_densenet121_TL.pt"),
    ("T3_best_TL_pth",    CKPT_DIR / "optimized_best_densenet121_TL.pth"),
    ("T4_champion_onnx",  ART_DIR  / "densenet121_tl_champion.onnx"),
    ("T4_champion_cfg",   ART_DIR  / "champion_config.json"),
    ("T7_retrained_onnx", ART_DIR  / "densenet121_tl_retrained_theme7.onnx"),
]

print("\n" + "="*90)
print("🔎 INVENTORY — Fichiers clés attendus")
print("="*90)
for name, path in candidates:
    print(f"{name:<20} {file_exists(path)}  {path}")

# Also list top checkpoints quickly
print("\n📦 Checkpoints disponibles (top 15):")
if CKPT_DIR.exists():
    for p in sorted(CKPT_DIR.glob("*"))[:15]:
        print(" -", p.name)
else:
    print("❌ CKPT_DIR absent")

print("\n🧪 Artifacts disponibles (top 15):")
if ART_DIR.exists():
    for p in sorted(ART_DIR.glob("*"))[:15]:
        print(" -", p.name)
else:
    print("❌ ART_DIR absent")

# --------------------------------------------
# 3) Load what we can (PyTorch)
# --------------------------------------------
loaded = {}

for tag, path in candidates:
    if path.suffix.lower() in [".pt", ".pth"] and path.exists():
        model, meta, load_res = load_model_from_path(path)
        loaded[tag] = {"path": str(path), "meta": meta, "load": load_res}
        print("\n" + "-"*90)
        print(f"📌 Loaded PyTorch checkpoint: {tag}  ({path.name})")
        print("Meta:", meta)
        print("Load:", load_res)

# --------------------------------------------
# 4) Read champion_config.json if present
# --------------------------------------------
champion_cfg = None
cfg_path = ART_DIR / "champion_config.json"
if cfg_path.exists():
    try:
        champion_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
        print("\n" + "-"*90)
        print("🧾 champion_config.json (résumé)")
        for k in ["model_name","input_size","num_classes","best_metric","best_value","onnx_path","timestamp"]:
            if k in champion_cfg:
                print(f" - {k}: {champion_cfg[k]}")
    except Exception as e:
        print("⚠️ Impossible de lire champion_config.json:", e)

# --------------------------------------------
# 5) Timeline narrative (Theme 2->3->4->7) + Report d'étonnement (auto)
# --------------------------------------------
print("\n" + "="*90)
print("🧭 TIMELINE (preuve itérative)")
print("="*90)

timeline = [
    ("Theme 2", "Baseline training pipeline + TensorBoard + checkpoints"),
    ("Theme 3", "Optimization / TL DenseNet121 (1ch) + best checkpoint saved"),
    ("Theme 4", "Champion export ONNX + validation PT vs ORT + benchmarking"),
    ("Theme 6", "Drift injected physically (noise/blur/rot/brightness/cutout)"),
    ("Theme 7", "Retraining on drifted data + new ONNX exported"),
    ("Theme 8", "Synthesis + decision framework + canary progressive rollout"),
]
for t, d in timeline:
    print(f"- {t:<8} : {d}")

print("\n" + "="*90)
print("🧠 RAPPORT D'ÉTONNEMENT / LESSONS LEARNED (auto-généré)")
print("="*90)

lessons = []

# 1) checkpoint formats
if (CKPT_DIR / "history_densenet121_TL.pt").exists():
    info = loaded.get("T3_history_TL_pt", {}).get("meta", {})
    if info:
        lessons.append("Les fichiers .pt peuvent contenir un dict 'historique' (pas directement un state_dict) : "
                       "il faut extraire correctement les poids (state_dict) avant chargement.")
# 2) best checkpoint
if (CKPT_DIR / "optimized_best_densenet121_TL.pth").exists():
    info = loaded.get("T3_best_TL_pth", {}).get("load", {})
    if info and info.get("missing", 999) == 0 and info.get("unexpected", 999) == 0:
        lessons.append("Le checkpoint .pth 'optimized_best...' est un state_dict propre (strict=True OK), "
                       "donc c'est la base la plus fiable pour fine-tuning et export ONNX.")
# 3) grayscale adaptation
lessons.append("Le dataset ImageFolder charge en RGB par défaut → nécessité de forcer Grayscale(1) pour un modèle 1 canal.")
# 4) reproducibility
lessons.append("La seed (42) doit être fixée partout (split, dataloader, entraînement) pour garantir des résultats reproductibles.")
# 5) onnx pipeline
if (ART_DIR / "densenet121_tl_champion.onnx").exists():
    lessons.append("Exporter en ONNX nécessite une validation numérique (max_abs_diff + top1 agreement) avant production.")
# 6) drift
lessons.append("Le drift peut provoquer une chute majeure de performance (ex: -10 points) même si le modèle est très bon en clean.")
# 7) safe deployment
lessons.append("Un déploiement canary progressif (10→20→50→100) permet d'éviter un remplacement risqué et de monitorer les régressions.")

for i, l in enumerate(lessons, 1):
    print(f"{i}. {l}")

print("\n✅ Story loader terminé.")


📁 WORK_DIR : C:\Users\Yanis\IA_GEN_PROJET
📦 CKPT_DIR : C:\Users\Yanis\IA_GEN_PROJET\checkpoints (exists=True)
🧪 ART_DIR  : C:\Users\Yanis\IA_GEN_PROJET\artifacts (exists=True)

🔎 INVENTORY — Fichiers clés attendus
T3_history_TL_pt     ✅  C:\Users\Yanis\IA_GEN_PROJET\checkpoints\history_densenet121_TL.pt
T3_best_TL_pth       ✅  C:\Users\Yanis\IA_GEN_PROJET\checkpoints\optimized_best_densenet121_TL.pth
T4_champion_onnx     ✅  C:\Users\Yanis\IA_GEN_PROJET\artifacts\densenet121_tl_champion.onnx
T4_champion_cfg      ✅  C:\Users\Yanis\IA_GEN_PROJET\artifacts\champion_config.json
T7_retrained_onnx    ✅  C:\Users\Yanis\IA_GEN_PROJET\artifacts\densenet121_tl_retrained_theme7.onnx

📦 Checkpoints disponibles (top 15):
 - baseline_metrics.pt
 - baseline_production_metrics.pt
 - baseline_theme-5_production_metrics.pt
 - best_model.pth
 - densenet121_retraining_work.pt
 - densenet121_to_finetune.pt
 - history_densenet121_TL.pt
 - history_opt.pt
 - history_resnet18_TL.pt
 - last_model.pth
 - optimize

In [2]:
# ========================================================================================================
# THEME 3 — CELLULE T3-STORY-1 : Model Registry Loader (auto) — PyTorch + ONNX
# ========================================================================================================

import json
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from collections import OrderedDict
import pandas as pd

WORK_DIR = Path(r"C:\Users\Yanis\IA_GEN_PROJET")
CKPT_DIR = WORK_DIR / "checkpoints"
ART_DIR  = WORK_DIR / "artifacts"

assert CKPT_DIR.exists(), f"❌ {CKPT_DIR} introuvable"
assert ART_DIR.exists(),  f"❌ {ART_DIR} introuvable"

def build_densenet121_1ch_2cls():
    m = models.densenet121(weights=None)
    m.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    m.classifier = nn.Linear(m.classifier.in_features, 2)
    return m

def detect_state_dict(obj):
    # returns (state_dict, mode)
    if isinstance(obj, OrderedDict):
        return obj, "OrderedDict(state_dict)"
    if isinstance(obj, dict):
        for k in ["model_state_dict", "state_dict"]:
            if k in obj and isinstance(obj[k], (dict, OrderedDict)):
                return obj[k], f"dict(checkpoint:{k})"
        # fallback: keep only tensor-like keys
        blacklist = {"epoch","phase","train_loss","train_acc","val_loss","val_acc",
                     "optimizer_state_dict","metrics","history","scheduler","scaler"}
        sd = {k:v for k,v in obj.items() if k not in blacklist and hasattr(v, "shape")}
        if len(sd) > 10:
            return sd, "dict(assumed_state_dict_filtered)"
    return None, "unknown"

def load_and_probe_pytorch(path: Path):
    try:
        obj = torch.load(path, map_location="cpu")
    except Exception as e:
        return {"load_ok": False, "error": repr(e), "format": "unreadable"}

    sd, fmt = detect_state_dict(obj)
    out = {"load_ok": True, "format": fmt, "type": str(type(obj))}
    if sd is None:
        out.update({"strict_ok": False, "missing": None, "unexpected": None})
        return out

    m = build_densenet121_1ch_2cls()
    try:
        m.load_state_dict(sd, strict=True)
        out.update({"strict_ok": True, "missing": 0, "unexpected": 0})
    except Exception:
        res = m.load_state_dict(sd, strict=False)
        out.update({
            "strict_ok": False,
            "missing": len(res.missing_keys),
            "unexpected": len(res.unexpected_keys),
            "missing_preview": res.missing_keys[:5],
            "unexpected_preview": res.unexpected_keys[:5]
        })
    return out

def guess_theme(name: str):
    n = name.lower()
    if "baseline" in n and "production" not in n:
        return "Theme 1/2 (baseline)"
    if "history_" in n:
        return "Theme 3 (training history)"
    if "optimized_best_densenet121" in n:
        return "Theme 3 (best TL DenseNet)"
    if "optimized_best_resnet18" in n:
        return "Theme 3 (best TL ResNet18)"
    if "retrain" in n or "retraining" in n or "theme7" in n:
        return "Theme 7 (retraining)"
    if "best_model" in n or "last_model" in n:
        return "Theme 2/3 (generic best/last)"
    return "Other"

rows = []

# --- scan checkpoints ---
for p in sorted(CKPT_DIR.glob("*")):
    if p.suffix.lower() in [".pt", ".pth"]:
        probe = load_and_probe_pytorch(p)
        rows.append({
            "name": p.name,
            "kind": "pytorch",
            "theme_guess": guess_theme(p.name),
            "path": str(p),
            "format": probe.get("format"),
            "strict_ok": probe.get("strict_ok"),
            "missing": probe.get("missing"),
            "unexpected": probe.get("unexpected"),
            "load_ok": probe.get("load_ok"),
        })

# --- scan artifacts (onnx + json config) ---
for p in sorted(ART_DIR.glob("*")):
    if p.suffix.lower() == ".onnx":
        rows.append({
            "name": p.name,
            "kind": "onnx",
            "theme_guess": "Theme 4/7 (deployment)",
            "path": str(p),
            "format": "onnx",
            "strict_ok": None,
            "missing": None,
            "unexpected": None,
            "load_ok": True,
        })
    if p.name == "champion_config.json":
        rows.append({
            "name": p.name,
            "kind": "config",
            "theme_guess": "Theme 4 (champion metadata)",
            "path": str(p),
            "format": "json",
            "strict_ok": None,
            "missing": None,
            "unexpected": None,
            "load_ok": True,
        })

df_registry = pd.DataFrame(rows).sort_values(["kind","theme_guess","name"]).reset_index(drop=True)

print("✅ Registry généré — artefacts détectés :", len(df_registry))
display(df_registry)

# Mini focus sur les artefacts principaux
key_files = [
    "history_densenet121_TL.pt",
    "optimized_best_densenet121_TL.pth",
    "densenet121_tl_champion.onnx",
    "densenet121_tl_retrained_theme7.onnx",
    "champion_config.json",
]
print("\n🎯 Key artifacts:")
display(df_registry[df_registry["name"].isin(key_files)].reset_index(drop=True))


✅ Registry généré — artefacts détectés : 17


,name,kind,theme_guess,path,format,strict_ok,missing,unexpected,load_ok
0,champion_config.json,config,Theme 4 (champion metadata),C:\Users\Yanis\IA_GEN_PROJET\artifacts\champio...,json,None,NaN,NaN,True
1,densenet121_tl_champion.onnx,onnx,Theme 4/7 (deployment),C:\Users\Yanis\IA_GEN_PROJET\artifacts\densene...,onnx,None,NaN,NaN,True
2,densenet121_tl_retrained_theme7.onnx,onnx,Theme 4/7 (deployment),C:\Users\Yanis\IA_GEN_PROJET\artifacts\densene...,onnx,None,NaN,NaN,True
3,baseline_production_metrics.pt,pytorch,Other,C:\Users\Yanis\IA_GEN_PROJET\checkpoints\basel...,unknown,False,NaN,NaN,True
4,baseline_theme-5_production_metrics.pt,pytorch,Other,C:\Users\Yanis\IA_GEN_PROJET\checkpoints\basel...,unknown,False,NaN,NaN,True
5,densenet121_to_finetune.pt,pytorch,Other,C:\Users\Yanis\IA_GEN_PROJET\checkpoints\dense...,unknown,False,NaN,NaN,True
6,optimized_best.pth,pytorch,Other,C:\Users\Yanis\IA_GEN_PROJET\checkpoints\optim...,OrderedDict(state_dict),False,606.0,122.0,True
7,baseline_metrics.pt,pytorch,Theme 1/2 (baseline),C:\Users\Yanis\IA_GEN_PROJET\checkpoints\basel...,unknown,False,NaN,NaN,True
8,best_model.pth,pytorch,Theme 2/3 (generic best/last),C:\Users\Yanis\IA_GEN_PROJET\checkpoints\best_...,OrderedDict(state_dict),False,606.0,25.0,True
9,last_model.pth,pytorch,Theme 2/3 (generic best/last),C:\Users\Yanis\IA_GEN_PROJET\checkpoints\last_...,OrderedDict(state_dict),False,606.0,25.0,True



🎯 Key artifacts:


,name,kind,theme_guess,path,format,strict_ok,missing,unexpected,load_ok
0,champion_config.json,config,Theme 4 (champion metadata),C:\Users\Yanis\IA_GEN_PROJET\artifacts\champio...,json,None,NaN,NaN,True
1,densenet121_tl_champion.onnx,onnx,Theme 4/7 (deployment),C:\Users\Yanis\IA_GEN_PROJET\artifacts\densene...,onnx,None,NaN,NaN,True
2,densenet121_tl_retrained_theme7.onnx,onnx,Theme 4/7 (deployment),C:\Users\Yanis\IA_GEN_PROJET\artifacts\densene...,onnx,None,NaN,NaN,True
3,optimized_best_densenet121_TL.pth,pytorch,Theme 3 (best TL DenseNet),C:\Users\Yanis\IA_GEN_PROJET\checkpoints\optim...,OrderedDict(state_dict),True,0.0,0.0,True
4,history_densenet121_TL.pt,pytorch,Theme 3 (training history),C:\Users\Yanis\IA_GEN_PROJET\checkpoints\histo...,unknown,False,NaN,NaN,True


In [3]:
# ========================================================================================================
# THEME 3 — CELLULE T3-STORY-2 : Narration itérative + Lessons learned + preuves (auto)
# ========================================================================================================

from pathlib import Path
import json

WORK_DIR = Path(r"C:\Users\Yanis\IA_GEN_PROJET")
CKPT_DIR = WORK_DIR / "checkpoints"
ART_DIR  = WORK_DIR / "artifacts"

# --- Artefacts clés
HIST_TL = CKPT_DIR / "history_densenet121_TL.pt"
BEST_TL = CKPT_DIR / "optimized_best_densenet121_TL.pth"
CHAMP_ONNX = ART_DIR / "densenet121_tl_champion.onnx"
RETR_ONNX  = ART_DIR / "densenet121_tl_retrained_theme7.onnx"
CFG        = ART_DIR / "champion_config.json"

print("="*110)
print("🧾 NARRATION (à copier dans le rapport) — Itérations & preuves par fichiers")
print("="*110)

print(f"""
### Traçabilité des itérations (preuves)
- **Theme 3 (Training / TL DenseNet121)** :
  - Historique d'entraînement : `{HIST_TL.name}` (checkpoint de type 'history', pas toujours un state_dict direct)
  - Meilleur modèle TL (réutilisable / strict state_dict) : `{BEST_TL.name}`

- **Theme 4 (Champion + déploiement)** :
  - Export ONNX du champion : `{CHAMP_ONNX.name}`
  - Métadonnées champion : `{CFG.name}` (nom, métrique cible, timestamp, etc.)

- **Theme 7 (Retraining)** :
  - Export ONNX du modèle retrainé : `{RETR_ONNX.name}`
""")

# champion config
if CFG.exists():
    try:
        cfg = json.loads(CFG.read_text(encoding="utf-8"))
        print("📌 Extrait champion_config.json :")
        for k in ["model_name","best_metric","best_value","timestamp","onnx_path"]:
            if k in cfg:
                print(f" - {k}: {cfg[k]}")
    except Exception as e:
        print("⚠️ Lecture champion_config.json impossible:", e)

print("\n" + "="*110)
print("🧠 RAPPORT D'ÉTONNEMENT / LEÇONS APPRISES (structuré)")
print("="*110)

print("""
1) **Format des checkpoints (.pt vs .pth)**
   - Un fichier `.pt` peut contenir un dictionnaire d'historique (logs + métriques + éventuellement poids).
   - Un fichier `.pth` est souvent un `state_dict` propre et directement réutilisable.
   → Conséquence : pour industrialiser, il faut standardiser le format de sauvegarde.

2) **Cohérence des entrées (1 canal)**
   - ImageFolder charge des images en RGB par défaut, alors que nos modèles ont été adaptés en 1 canal (Grayscale).
   → Conséquence : sans `Grayscale(1)` + normalisation 1 canal, l’inférence casse (shape mismatch) ou dégrade.

3) **Reproductibilité**
   - Le split 80/20 et le fine-tuning varient sans seed fixe.
   → Conséquence : seed=42 partout (split, dataloader, torch) est indispensable pour comparer des itérations.

4) **Drift = casse de performance**
   - Un modèle excellent en clean peut chuter fortement en drift (observé ensuite au Theme 6).
   → Conséquence : on ne peut pas “entraîner une fois et oublier” : monitoring + déclenchement retraining.

5) **Déploiement sûr**
   - Export ONNX + validation numérique (écart PT vs ORT) sont nécessaires avant prod.
   - Canary progressif (10→20→50→100) réduit le risque et documente la décision.
""")

print("\n✅ Narration + étonnement générés.")


🧾 NARRATION (à copier dans le rapport) — Itérations & preuves par fichiers

### Traçabilité des itérations (preuves)
- **Theme 3 (Training / TL DenseNet121)** :
  - Historique d'entraînement : `history_densenet121_TL.pt` (checkpoint de type 'history', pas toujours un state_dict direct)
  - Meilleur modèle TL (réutilisable / strict state_dict) : `optimized_best_densenet121_TL.pth`

- **Theme 4 (Champion + déploiement)** :
  - Export ONNX du champion : `densenet121_tl_champion.onnx`
  - Métadonnées champion : `champion_config.json` (nom, métrique cible, timestamp, etc.)

- **Theme 7 (Retraining)** :
  - Export ONNX du modèle retrainé : `densenet121_tl_retrained_theme7.onnx`

📌 Extrait champion_config.json :
 - model_name: DenseNet121_TL

🧠 RAPPORT D'ÉTONNEMENT / LEÇONS APPRISES (structuré)

1) **Format des checkpoints (.pt vs .pth)**
   - Un fichier `.pt` peut contenir un dictionnaire d'historique (logs + métriques + éventuellement poids).
   - Un fichier `.pth` est souvent un `state_dic